# 00 · Setup and runtime check

Run once per new Colab runtime type, or whenever something breaks.
Checks the hardware, installs extras, confirms gated Hugging Face access, and logs exact package versions to Drive.

**Before running:** in Colab, open the key icon (Secrets) and add `HF_TOKEN` with a *read* token from
huggingface.co/settings/tokens, and allow notebook access. You must have accepted the dataset's access conditions on its Hugging Face page.

In [ ]:
# Bootstrap: same cell in every notebook. On Colab: mount Drive, get the repo, install extras.
import os, sys, subprocess
REPO_URL = "https://github.com/sanjanasinghx/dream2image-es606.git"
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/dream2image-es606"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-colab.txt"], check=True)
else:
    REPO = os.path.abspath("..")
sys.path.insert(0, REPO)
from d2i import env
cfg = env.load_config()
rng = env.seed_everything(cfg["seed"])
print("data root:", env.data_root())

In [ ]:
# Hardware
import psutil, shutil
print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB, CPUs: {os.cpu_count()}")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True) if shutil.which("nvidia-smi") else None
print("GPU:", gpu.stdout.strip() if gpu and gpu.returncode == 0 else "none (fine for NB1-NB3)")
print(f"Drive free: {shutil.disk_usage(env.data_root()).free / 1e9:.1f} GB")

In [ ]:
# Gated dataset access
from huggingface_hub import HfApi
api = HfApi()
api.auth_check(cfg["dataset"]["hf_id"], repo_type="dataset", token=env.hf_token())
files = [f for f in api.list_repo_files(cfg["dataset"]["hf_id"], repo_type="dataset", token=env.hf_token())]
print("access OK; files:", files)

In [ ]:
# Log versions
import importlib, json, platform
pkgs = ["numpy", "scipy", "pandas", "sklearn", "mne", "yasa", "specparam", "torch",
        "transformers", "pyarrow", "huggingface_hub", "matplotlib"]
versions = {"python": platform.python_version()}
for p in pkgs:
    try:
        versions[p] = importlib.import_module(p).__version__
    except Exception as e:
        versions[p] = f"MISSING ({type(e).__name__})"
print(json.dumps(versions, indent=1))
out = env.path(cfg, "results") / "logs" / "environment.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({"versions": versions, "gpu": gpu.stdout.strip() if gpu else None}, indent=2))
print("wrote", out)